# 04 - Official Denormalized Open-Loop Evaluation

K-step Euler ODE inference. MSE/MAE/RMSE tính trên **raw denormalized action space**.

In [ ]:
!pip install -q pandas pyarrow numpy tqdm matplotlib

In [ ]:
from pathlib import Path
import json, math, time
import numpy as np, pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE_TEST = False; ACTION_HORIZON = 16; STATE_HORIZON = 1; K = 4
MAX_EVAL_SAMPLES = 2048 if SMOKE_TEST else None
BATCH_SIZE = 512; NUM_WORKERS = 2; USE_AMP = True
OUTPUT_ROOT = Path("/kaggle/working/gr00t_official_eval"); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
def find_dir(name, required):
    cand = []
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if base.exists(): cand.extend(base.rglob(name))
    for c in sorted(cand, key=str):
        if c.is_dir() and all((c / r).exists() for r in required): return c
    raise FileNotFoundError(name)
PREPARED_ROOT = find_dir(f"gr00t_prepared_official_H{ACTION_HORIZON}", ["states_test.npy", "actions_test_chunk.npy", "action_mask_test.npy", "normalization_stats.json"])
RUN_ROOT = find_dir("official_mini_vldit_H16", ["checkpoint_last.pt", "config.json", "train_summary.json"])
FEATURE_ROOT = RUN_ROOT.parent
ckpt = torch.load(RUN_ROOT / "checkpoint_last.pt", map_location=device); cfg = ckpt["config"]; stats = ckpt["normalization"]
print("PREPARED_ROOT:", PREPARED_ROOT); print("RUN_ROOT:", RUN_ROOT)

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, hidden_dim, buckets=1000):
        super().__init__()
        self.buckets = buckets
        self.embed = nn.Embedding(buckets, hidden_dim)
        self.mlp = nn.Sequential(nn.SiLU(), nn.Linear(hidden_dim, hidden_dim))
    def forward(self, t):
        idx = torch.clamp((t * self.buckets).long(), 0, self.buckets - 1)
        return self.mlp(self.embed(idx))

class CrossBlock(nn.Module):
    def __init__(self, hidden_dim, heads, dropout):
        super().__init__()
        self.n1 = nn.LayerNorm(hidden_dim)
        self.sa = nn.MultiheadAttention(hidden_dim, heads, dropout=dropout, batch_first=True)
        self.n2 = nn.LayerNorm(hidden_dim)
        self.ca = nn.MultiheadAttention(hidden_dim, heads, dropout=dropout, batch_first=True)
        self.n3 = nn.LayerNorm(hidden_dim)
        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim), nn.Dropout(dropout)
        )
    def forward(self, x, mem):
        y = self.n1(x)
        x = x + self.sa(y, y, y, need_weights=False)[0]
        y = self.n2(x)
        x = x + self.ca(y, mem, mem, need_weights=False)[0]
        return x + self.ff(self.n3(x))

class OfficialMiniVLDiT(nn.Module):
    def __init__(self, state_dim, action_dim, state_horizon, action_horizon, vl_dim,
                 hidden_dim=512, layers=8, heads=8, dropout=0.1, buckets=1000):
        super().__init__()
        self.state_horizon = state_horizon
        self.action_horizon = action_horizon
        self.state_proj = nn.Linear(state_dim, hidden_dim)
        self.action_proj = nn.Linear(action_dim, hidden_dim)
        self.vl_proj = nn.Linear(vl_dim, hidden_dim)
        self.time = TimeEmbedding(hidden_dim, buckets)
        self.state_pos = nn.Embedding(state_horizon, hidden_dim)
        self.action_pos = nn.Embedding(action_horizon, hidden_dim)
        self.blocks = nn.ModuleList([CrossBlock(hidden_dim, heads, dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(hidden_dim)
        self.out = nn.Linear(hidden_dim, action_dim)
    def forward(self, noisy_action, state_history, vl_features, t):
        s_pos = self.state_pos(torch.arange(self.state_horizon, device=noisy_action.device))[None]
        a_pos = self.action_pos(torch.arange(self.action_horizon, device=noisy_action.device))[None]
        s = self.state_proj(state_history) + s_pos
        a = self.action_proj(noisy_action) + a_pos
        x = torch.cat([s, a], dim=1) + self.time(t)[:, None, :]
        mem = self.vl_proj(vl_features)
        for block in self.blocks:
            x = block(x, mem)
        return self.out(self.norm(x[:, self.state_horizon:]))

model = OfficialMiniVLDiT(44, 44, cfg["state_horizon"], cfg["action_horizon"], cfg["vl_feature_dim"], cfg["hidden_dim"], cfg["num_layers"], cfg["num_heads"], cfg["dropout"]).to(device)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()

In [ ]:
class EvalDS(Dataset):
    def __init__(self):
        self.states = np.load(PREPARED_ROOT / "states_test.npy", mmap_mode="r")
        self.actions = np.load(PREPARED_ROOT / "actions_test_chunk.npy", mmap_mode="r")
        self.masks = np.load(PREPARED_ROOT / "action_mask_test.npy", mmap_mode="r")
        self.features = np.load(FEATURE_ROOT / "vl_features_test.npy", mmap_mode="r")
        self.index = pd.read_parquet(FEATURE_ROOT / "vl_feature_index_test.parquet").sort_values("feature_index").reset_index(drop=True)
        if MAX_EVAL_SAMPLES is not None and len(self.index) > MAX_EVAL_SAMPLES:
            self.index = self.index.sample(n=int(MAX_EVAL_SAMPLES), random_state=42).reset_index(drop=True)
        self.sm = np.asarray(stats["state_mean"], np.float32); self.ss = np.asarray(stats["state_std"], np.float32)
        self.am = np.asarray(stats["action_mean"], np.float32); self.asd = np.asarray(stats["action_std"], np.float32)
    def __len__(self): return len(self.index)
    def __getitem__(self, i):
        r = self.index.iloc[i]; sid = int(r.sample_id); fid = int(r.feature_index)
        state = (np.asarray(self.states[sid], np.float32) - self.sm) / self.ss
        action = (np.asarray(self.actions[sid], np.float32) - self.am) / self.asd
        return {"state": torch.from_numpy(state), "action": torch.from_numpy(action), "mask": torch.from_numpy(np.asarray(self.masks[sid], np.float32)),
                "vl": torch.from_numpy(np.asarray(self.features[fid], np.float32)), "subset": r.subset}
def collate(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in ["state","action","mask","vl"]} | {"subset": [b["subset"] for b in batch]}
ds = EvalDS(); loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), collate_fn=collate)
am = torch.tensor(ds.am, device=device)[None,None,:]; ast = torch.tensor(ds.asd, device=device)[None,None,:]
sm = torch.tensor(ds.sm, device=device)[None,None,:]; ss = torch.tensor(ds.ss, device=device)[None,None,:]
print("eval samples:", len(ds))

In [ ]:
@torch.no_grad()
def sample(state, vl, steps):
    x = torch.randn(state.size(0), ACTION_HORIZON, 44, device=state.device, dtype=state.dtype)
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((state.size(0),), k / steps, device=state.device, dtype=state.dtype)
        x = x + dt * model(x, state, vl, t)
    return x

def bucket(): return {"sq":0.0, "ab":0.0, "count":0, "samples":0}
def add(b, sq, ab, m, samples): b["sq"] += float(sq.sum()); b["ab"] += float(ab.sum()); b["count"] += int(m.sum()); b["samples"] += int(samples)
def fin(b):
    mse = b["sq"] / max(b["count"], 1); mae = b["ab"] / max(b["count"], 1)
    return {"samples": int(b["samples"]), "values": int(b["count"]), "raw_mse": mse, "raw_mae": mae, "raw_rmse": math.sqrt(mse)}

overall = bucket(); baseline_mean = bucket(); baseline_last_state = bucket()
per_subset = {}; per_h = [bucket() for _ in range(ACTION_HORIZON)]; per_dim = [bucket() for _ in range(44)]
flow_sum = 0.0; flow_batches = 0; preview_p = []; preview_t = []; start = time.time()
for b in tqdm(loader, desc="eval"):
    state = b["state"].to(device); action = b["action"].to(device); mask = b["mask"].to(device); vl = b["vl"].to(device)
    with torch.cuda.amp.autocast(enabled=USE_AMP and torch.cuda.is_available()):
        pred = sample(state, vl, K)
        t = torch.rand(action.size(0), device=device, dtype=action.dtype); noise = torch.randn_like(action); target = action - noise
        flow = ((model((1-t[:,None,None])*noise+t[:,None,None]*action, state, vl, t) - target).pow(2) * mask).sum() / (mask.sum()+1e-6)
    pred_raw = pred.float() * ast + am; target_raw = action.float() * ast + am
    diff = pred_raw - target_raw; sq = (diff.square() * mask).cpu().numpy(); ab = (diff.abs() * mask).cpu().numpy(); m = mask.cpu().numpy()
    add(overall, sq, ab, m, action.size(0))
    # Baseline 1: action trung binh train split trong raw space.
    mean_diff = am.expand_as(target_raw) - target_raw
    add(baseline_mean, (mean_diff.square() * mask).cpu().numpy(), (mean_diff.abs() * mask).cpu().numpy(), m, action.size(0))
    # Baseline 2: lap lai state cuoi cung lam action du doan. Chi dung de so sanh tham khao vi state/action cung 44 dim trong GR-1 subset.
    state_raw = state.float() * ss + sm
    last_state_pred = state_raw[:, -1:, :].expand_as(target_raw)
    state_diff = last_state_pred - target_raw
    add(baseline_last_state, (state_diff.square() * mask).cpu().numpy(), (state_diff.abs() * mask).cpu().numpy(), m, action.size(0))
    for subset in set(b["subset"]):
        ids = [i for i, s in enumerate(b["subset"]) if s == subset]
        add(per_subset.setdefault(subset, bucket()), sq[ids], ab[ids], m[ids], len(ids))
    for h in range(ACTION_HORIZON): add(per_h[h], sq[:,h,:], ab[:,h,:], m[:,h,:], action.size(0))
    for d in range(44): add(per_dim[d], sq[:,:,d], ab[:,:,d], m[:,:,d], action.size(0))
    flow_sum += float(flow.cpu()); flow_batches += 1
    if len(preview_p) < 512:
        take = min(512-len(preview_p), pred_raw.size(0)); preview_p += pred_raw[:take,0,0].cpu().tolist(); preview_t += target_raw[:take,0,0].cpu().tolist()

metrics = fin(overall); metrics["flow_matching_test_loss_normalized"] = flow_sum / max(flow_batches, 1); metrics["denoising_steps"] = K
compare_df = pd.DataFrame([
    {"model":"baseline_train_action_mean", "input":"none", "S":STATE_HORIZON, "H":ACTION_HORIZON, "K":0, **fin(baseline_mean)},
    {"model":"baseline_last_state_repeat", "input":"state", "S":STATE_HORIZON, "H":ACTION_HORIZON, "K":0, **fin(baseline_last_state)},
    {"model":"OfficialMiniVLDiT", "input":"state+video_language", "S":STATE_HORIZON, "H":ACTION_HORIZON, "K":K, **metrics},
])
subset_df = pd.DataFrame([{"subset": k, **fin(v)} for k, v in sorted(per_subset.items())])
h_df = pd.DataFrame([{"horizon_step": i, **fin(v)} for i, v in enumerate(per_h)])
dim_df = pd.DataFrame([{"action_dim": i, **fin(v)} for i, v in enumerate(per_dim)])
display(compare_df); display(subset_df); display(h_df.head())

In [ ]:
compare_df.to_csv(OUTPUT_ROOT/"eval_metrics.csv", index=False)
compare_df.to_csv(OUTPUT_ROOT/"baseline_vs_official_mini.csv", index=False)
subset_df.to_csv(OUTPUT_ROOT/"per_subset_metrics.csv", index=False); h_df.to_csv(OUTPUT_ROOT/"per_horizon_metrics.csv", index=False); dim_df.to_csv(OUTPUT_ROOT/"per_action_dim_metrics.csv", index=False)
plt.figure(figsize=(5,5)); plt.scatter(preview_t, preview_p, s=7, alpha=.35); mn=min(preview_t+preview_p); mx=max(preview_t+preview_p); plt.plot([mn,mx],[mn,mx],color="black"); plt.xlabel("GT raw action dim0"); plt.ylabel("Pred raw action dim0"); plt.tight_layout(); plt.savefig(OUTPUT_ROOT/"prediction_vs_ground_truth_raw.png", dpi=160); plt.show()
plt.figure(figsize=(7,4)); plt.plot(h_df.horizon_step, h_df.raw_mse, marker="o"); plt.xlabel("horizon step"); plt.ylabel("raw MSE"); plt.grid(True, alpha=.3); plt.tight_layout(); plt.savefig(OUTPUT_ROOT/"horizon_error_curve_raw.png", dpi=160); plt.show()
summary = {"status":"completed", "eval_samples": int(metrics["samples"]), "metrics": metrics, "comparison_table": compare_df.to_dict("records"), "denormalized_raw_action_metrics": True, "uses_action_mask": True, "elapsed_sec": round(time.time()-start, 2)}
(OUTPUT_ROOT/"eval_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))